# Extended Data Figures 2 and 9: Homogeneous DISC reputations and initial-reputation robustness

Run from the notebook directory. Saved equilibrium tables are used by default; set `RECOMPUTE=True` to regenerate the parameter planes. Extended Data Figure 2 shows the reputation drift of homogeneous DISC populations (panels a–d) and the population-level comparison of all-good and all-bad initial reputations (panel e). Extended Data Figure 9 shows the one-step readout plane and the full-readout depth plane (formerly rows 2–3 of Extended Data Figure 2). Dependencies: NumPy, pandas, SciPy and Matplotlib.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, Rectangle
from matplotlib.ticker import MaxNLocator, NullFormatter
from scipy.optimize import brentq
from IPython.display import display

ROOT = Path.cwd()
DATA = ROOT / 'data' / 'extended_data_figure2'
FIGURES = ROOT / 'figures' / 'extended_data_figure2'
DATA.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
OUTPUT = DATA
RECOMPUTE = False
NORMS = {'SC': (1, 0), 'SH': (0, 0), 'SJ': (0, 1), 'SS': (1, 1)}
NAMES = {'SC':'Scoring', 'SH':'Shunning', 'SJ':'Stern Judging', 'SS':'Simple Standing'}
E1 = E2 = .02
ALPHA = BETA = 1.
MULTI_DEPTHS = {'SC':(2,0), 'SH':(3,0), 'SJ':(4,3), 'SS':(1,2)}
MAX_DEPTH, ALPHA_BETA_GRID = 8, 161
ROOT_GRID, PLANE_ROOT_GRID = 2001, 401
STABILITY_TOL = 1e-7
COLORS = {'without':'#6B6B6B', 'one':'#0077BB', 'multi':'#D55E00'}
KINDS = ('without', 'one', 'multi')
CONFIG = dict(e1=E1,e2=E2,alpha=ALPHA,beta=BETA,
    multi_depths={n:list(v) for n,v in MULTI_DEPTHS.items()},max_depth=MAX_DEPTH,
    alpha_beta_grid=ALPHA_BETA_GRID,root_grid=ROOT_GRID,plane_root_grid=PLANE_ROOT_GRID,
    stability_tolerance=STABILITY_TOL)
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':9,'axes.labelsize':9,
    'xtick.labelsize':8,'ytick.labelsize':8,'axes.spines.top':False,'axes.spines.right':False,
    'axes.linewidth':.8,'figure.dpi':120,'pdf.fonttype':42,'svg.fonttype':'none','hatch.linewidth':.5})


## Stationary response and self-consistency

The scalar drift is an auxiliary equilibration flow. Its roots describe stationary reputation self-consistency, rather than the exact transient dynamics of all internal states.

In [2]:
def validate(k1, k2, alpha, beta, e1, e2):
    if any((isinstance(k, bool) or int(k) != k or (not 0 <= k <= 64) for k in (k1, k2))):
        raise ValueError('Depths must be integers in [0,64]; larger depths require precision checks.')
    if not all((np.isfinite(x) and 0 <= x <= 1 for x in (alpha, beta))):
        raise ValueError('alpha and beta must lie in [0,1].')
    if not all((np.isfinite(x) and 0 <= x < 0.5 for x in (e1, e2))):
        raise ValueError('Error probabilities must lie in [0,0.5).')

def transitions(k1, k2, alpha=1.0, beta=1.0):
    """State order: G, W1,...,Wk1, B, S1,...,Sk2. Always direct reset."""
    validate(k1, k2, alpha, beta, E1, E2)
    b = k1 + 1
    size = k1 + k2 + 2
    plus = np.zeros(size, dtype=int)
    minus = np.full(size, b, dtype=int)
    plus[b] = b + 1 if k2 else 0
    for m in range(k1 + 1):
        minus[m] = m + 1 if m < k1 else b
    for n in range(1, k2 + 1):
        plus[b + n] = b + n + 1 if n < k2 else 0
    sigma = np.r_[1.0, alpha ** np.arange(1, k1 + 1), 0.0, 1.0 - beta ** np.arange(1, k2 + 1)]
    return (np.eye(size)[plus], np.eye(size)[minus], sigma)

def stationary_distribution(phi, k1, k2):
    """Unnormalised stationary weights; no endpoint divisions by phi or 1-phi."""
    phi = np.asarray(phi, dtype=float)
    if np.any(~np.isfinite(phi)) or np.any((phi < 0) | (phi > 1)):
        raise ValueError('phi must be a probability.')
    v = 1.0 - phi
    weights = np.stack([phi ** (k2 + 1) * v ** m for m in range(k1 + 1)] + [v ** (k1 + 1) * phi ** n for n in range(k2 + 1)], axis=-1)
    return weights / weights.sum(axis=-1, keepdims=True)

def reputation_map(phi, k1, k2, alpha=1.0, beta=1.0):
    sigma = np.r_[1.0, alpha ** np.arange(1, k1 + 1), 0.0, 1.0 - beta ** np.arange(1, k2 + 1)]
    return np.sum(stationary_distribution(phi, k1, k2) * sigma, axis=-1)

def map_and_derivative(phi, k1, k2, alpha, beta):
    """Analytic dR/dphi, using the same stationary weights as R."""
    phi = np.asarray(phi, dtype=float)
    v = 1.0 - phi
    a, b = (phi ** (k2 + 1), v ** (k1 + 1))
    da, db = ((k2 + 1) * phi ** k2, -(k1 + 1) * v ** k1)
    weights, derivatives, readouts = ([], [], [])
    for m in range(k1 + 1):
        weights.append(a * v ** m)
        derivatives.append(da * v ** m - (a * m * v ** (m - 1) if m else 0.0))
        readouts.append(alpha ** m)
    for n in range(k2 + 1):
        weights.append(b * phi ** n)
        derivatives.append(db * phi ** n + (b * n * phi ** (n - 1) if n else 0.0))
        readouts.append(1.0 - beta ** n)
    weights = np.stack(weights, axis=-1)
    derivatives = np.stack(derivatives, axis=-1)
    z = weights.sum(axis=-1)
    good = np.sum(weights * np.asarray(readouts), axis=-1)
    dgood = np.sum(derivatives * np.asarray(readouts), axis=-1)
    r = good / z
    derivative = (dgood - r * derivatives.sum(axis=-1)) / z
    bad = np.sum(weights * (1.0 - np.asarray(readouts)), axis=-1) / z
    return (r, derivative, bad)

def assessment(r, norm, e1=E1, e2=E2):
    p, q = NORMS[norm]
    eps = (1.0 - e1) * (1.0 - e2) + e1 * e2
    cg, dg = (eps, e2)
    cb = p * (eps - e2) + q * (1.0 - eps - e2) + e2
    db = q * (1.0 - 2.0 * e2) + e2
    r = np.asarray(r, dtype=float)
    phi = r * r * cg + r * (1.0 - r) * (cb + dg) + (1.0 - r) ** 2 * db
    dphi = 2.0 * r * cg + (1.0 - 2.0 * r) * (cb + dg) - 2.0 * (1.0 - r) * db
    return (phi, dphi)

def drift_and_derivative(r, norm, k1, k2, alpha=ALPHA, beta=BETA, e1=E1, e2=E2):
    phi, dphi = assessment(r, norm, e1, e2)
    good, dr, bad = map_and_derivative(phi, k1, k2, alpha, beta)
    drift = np.where(np.asarray(r) > 0.5, 1.0 - np.asarray(r) - bad, good - r)
    return (drift, dr * dphi - 1.0)

def fixed_points(norm, k1, k2, alpha=ALPHA, beta=BETA, e1=E1, e2=E2, n_grid=ROOT_GRID):
    """Find extrema first, then bracket roots on monotone intervals; retain near-folds."""
    validate(k1, k2, alpha, beta, e1, e2)
    args = (norm, k1, k2, alpha, beta, e1, e2)
    f = lambda r: float(drift_and_derivative(r, *args)[0])
    df = lambda r: float(drift_and_derivative(r, *args)[1])
    grid = np.linspace(0.0, 1.0, n_grid)
    derivatives = drift_and_derivative(grid, *args)[1]
    critical = [float(grid[i]) for i in np.flatnonzero(derivatives == 0.0)]
    for i in np.flatnonzero(np.signbit(derivatives[:-1]) != np.signbit(derivatives[1:])):
        critical.append(brentq(df, grid[i], grid[i + 1], xtol=5e-15))
    bounds = sorted(set([0.0, 1.0] + critical))
    roots = [r for r in bounds if f(r) == 0.0]
    near_folds = [r for r in critical if abs(f(r)) < 5e-12]
    roots += near_folds
    for left, right in zip(bounds[:-1], bounds[1:]):
        if left in near_folds or right in near_folds:
            continue
        if np.signbit(f(left)) != np.signbit(f(right)) and f(left) != 0.0 and (f(right) != 0.0):
            roots.append(brentq(f, left, right, xtol=5e-16, rtol=1e-15))
    unique = []
    for r in sorted(roots):
        if not unique or abs(r - unique[-1]) > 1e-09:
            unique.append(r)
    rows = []
    for r in unique:
        slope = df(r)
        stability = 'stable' if slope < -STABILITY_TOL else 'unstable' if slope > STABILITY_TOL else 'near_fold'
        rows.append(dict(norm=norm, k1=k1, k2=k2, alpha=alpha, beta=beta, e1=e1, e2=e2, r=r, slope=slope, stability=stability, residual=abs(f(r)), rounded_boundary=bool(r in (0.0, 1.0) and f(r) != 0.0)))
    if not rows:
        raise RuntimeError('No root detected; refine the grid before interpreting this case.')
    return rows

def equilibrium_summary(rows):
    stable = sorted((row['r'] for row in rows if row['stability'] == 'stable'))
    unstable = sorted((row['r'] for row in rows if row['stability'] == 'unstable'))
    near = any((row['stability'] == 'near_fold' for row in rows))
    basin = 1.0 - max([u for u in unstable if stable and u < stable[-1]], default=0.0) if stable else np.nan
    if near:
        basin = np.nan
    return dict(n_stable=len(stable), low=min(stable, default=np.nan), high=max(stable, default=np.nan), high_basin_reduced=basin, near_fold=near, rounded_boundary=any((row['rounded_boundary'] for row in rows)))

def latent_linearization(row):
    """Full state-occupation ODE and Jacobian restricted to the probability simplex."""
    k1, k2 = (row['k1'], row['k2'])
    plus, minus, sigma = transitions(k1, k2, row['alpha'], row['beta'])
    phi, dphi = assessment(row['r'], row['norm'], row['e1'], row['e2'])
    x = stationary_distribution(phi, k1, k2)
    generator = (phi * plus + (1.0 - phi) * minus).T - np.eye(len(x))
    jacobian = generator + np.outer((plus - minus).T @ x, dphi * sigma)
    tangent = jacobian[:-1, :-1] - jacobian[:-1, -1, None]
    eigenvalues = np.linalg.eigvals(tangent)
    dominant = eigenvalues[np.argmax(eigenvalues.real)]
    return dict(full_max_real=float(dominant.real), full_dominant_imag=float(abs(dominant.imag)), full_stability='stable' if dominant.real < -STABILITY_TOL else 'unstable' if dominant.real > STABILITY_TOL else 'near_fold', state_residual=float(np.max(np.abs(generator @ x))))

def self_check():
    phi = np.linspace(0.0, 1.0, 501)
    for a, b in [(0, 0), (1, 0), (0, 1), (1, 1), (0.7, 0.9)]:
        original = (phi ** 2 * (1 + a * (1 - phi)) + (1 - b) * phi * (1 - phi) ** 2) / (1 - phi + phi ** 2)
        assert np.allclose(reputation_map(phi, 1, 1, a, b), original, atol=1e-14)
    assert np.allclose(reputation_map(phi, 0, 0), phi, atol=1e-14)
    rng = np.random.default_rng(20261005)
    for _ in range(20):
        k1, k2 = map(int, rng.integers(0, 9, size=2))
        a, b, z = rng.uniform(0.05, 0.95, size=3)
        plus, minus, sigma = transitions(k1, k2, a, b)
        system = (z * plus + (1 - z) * minus).T - np.eye(len(sigma))
        system[-1] = 1.0
        rhs = np.zeros(len(sigma))
        rhs[-1] = 1.0
        stationary = np.linalg.solve(system, rhs)
        assert np.allclose(stationary, stationary_distribution(z, k1, k2), atol=1e-10)
        assert np.isclose(stationary @ sigma, reputation_map(z, k1, k2, a, b), atol=1e-10)
    for norm, pair in MULTI_DEPTHS.items():
        validate(*pair, 1.0, 1.0, E1, E2)
        assert max(pair) <= MAX_DEPTH, 'Increase MAX_DEPTH to include every marked multi-step pair.'
        if E1 == 0.02 and E2 == 0.02:
            expected = 1 if norm == 'SS' else 2
            assert equilibrium_summary(fixed_points(norm, 1, 1))['n_stable'] == expected
        for a, b in [(0, 0), (1, 1), (0.5, 0.5), (0.9, 1.0)]:
            coarse = fixed_points(norm, 1, 1, a, b, n_grid=PLANE_ROOT_GRID)
            fine = fixed_points(norm, 1, 1, a, b, n_grid=4001)
            assert [r['stability'] for r in coarse] == [r['stability'] for r in fine]
            assert np.allclose([r['r'] for r in coarse], [r['r'] for r in fine], atol=1e-09)
        for k1, k2 in [(0, 0), (1, 1), pair]:
            for row in fixed_points(norm, k1, k2):
                assert row['residual'] < 1e-10
                assert latent_linearization(row)['full_stability'] == row['stability']
    return {'one_step_formula': 'passed', 'random_markov_chains': 20, 'root_resolution': 'passed', 'selected_full_state_stability': 'passed'}

def condition_pairs(norm):
    return ((0, 0), (1, 1), MULTI_DEPTHS[norm])

def compute_planes():
    values = np.linspace(0.0, 1.0, ALPHA_BETA_GRID)
    readout_planes, depth_planes = ({}, {})
    readout_records, depth_records, roots_records = ([], [], [])
    for norm in NORMS:
        readout_high = np.full((len(values), len(values)), np.nan)
        readout_count = np.zeros(readout_high.shape, dtype=int)
        readout_near = np.zeros(readout_high.shape, dtype=bool)
        for j, beta in enumerate(values):
            for i, alpha in enumerate(values):
                rows = fixed_points(norm, 1, 1, alpha, beta, n_grid=PLANE_ROOT_GRID)
                summary = equilibrium_summary(rows)
                readout_high[j, i] = summary['high'] if not summary['near_fold'] else np.nan
                readout_count[j, i] = summary['n_stable']
                readout_near[j, i] = summary['near_fold']
                readout_records.append(dict(norm=norm, alpha=alpha, beta=beta, **summary))
        readout_planes[norm] = dict(high=readout_high, count=readout_count, near=readout_near)
        depth_high = np.full((MAX_DEPTH + 1, MAX_DEPTH + 1), np.nan)
        depth_count = np.zeros(depth_high.shape, dtype=int)
        depth_near = np.zeros(depth_high.shape, dtype=bool)
        for k2 in range(MAX_DEPTH + 1):
            for k1 in range(MAX_DEPTH + 1):
                rows = fixed_points(norm, k1, k2)
                for row in rows:
                    row.update(latent_linearization(row))
                    assert row['full_stability'] == row['stability'], 'Reduced/full stability mismatch.'
                    row['intended_cooperation'] = row['r']
                    row['realized_cooperation'] = (1 - E1) * row['r']
                roots_records.extend(rows)
                summary = equilibrium_summary(rows)
                depth_high[k2, k1] = summary['high'] if not summary['near_fold'] else np.nan
                depth_count[k2, k1] = summary['n_stable']
                depth_near[k2, k1] = summary['near_fold']
                depth_records.append(dict(norm=norm, k1=k1, k2=k2, **summary))
        depth_planes[norm] = dict(high=depth_high, count=depth_count, near=depth_near)
        print(norm, 'readout/depth planes finished', flush=True)
    pd.DataFrame(readout_records).to_csv(OUTPUT / 'alpha_beta_summary.csv', index=False)
    pd.DataFrame(depth_records).to_csv(OUTPUT / 'k1_k2_summary.csv', index=False)
    pd.DataFrame(roots_records).to_csv(OUTPUT / 'k1_k2_all_equilibria.csv', index=False)
    assert all((np.nanmin(x['high']) >= 0 and np.nanmax(x['high']) <= 1 for x in list(readout_planes.values()) + list(depth_planes.values())))
    assert max((x['count'].max() for x in list(readout_planes.values()) + list(depth_planes.values()))) <= 2, 'More than two stable roots: extend the stability legend before plotting.'
    rate_roots = {norm: [fixed_points(norm, *pair) for pair in condition_pairs(norm)] for norm in NORMS}
    table = []
    for norm in NORMS:
        for kind, pair, rows in zip(KINDS, condition_pairs(norm), rate_roots[norm]):
            for row in rows:
                table.append(dict(condition=kind, **row, intended_cooperation=row['r'], realized_cooperation=(1 - E1) * row['r']))
    selected_equilibria = pd.DataFrame(table)
    selected_equilibria.to_csv(OUTPUT / 'three_curve_equilibria.csv', index=False)
    return (values, readout_planes, depth_planes, rate_roots, selected_equilibria)

In [3]:
paths = {name: DATA / name for name in ('three_curve_equilibria.csv', 'alpha_beta_summary.csv',
    'k1_k2_summary.csv', 'k1_k2_all_equilibria.csv')}
config_file = DATA / 'input_config.json'
if RECOMPUTE:
    values, readout_planes, depth_planes, rate_roots, selected_equilibria = compute_planes()
    config_file.write_text(json.dumps(CONFIG, indent=2) + '\n')
else:
    if not config_file.is_file() or not all(p.is_file() for p in paths.values()):
        raise FileNotFoundError('Supply data/extended_data_figure2 or set RECOMPUTE=True.')
    if json.loads(config_file.read_text()) != CONFIG:
        raise ValueError('Saved configuration differs from the requested parameters.')
    selected_equilibria = pd.read_csv(paths['three_curve_equilibria.csv'])
    readout_table = pd.read_csv(paths['alpha_beta_summary.csv'])
    depth_table = pd.read_csv(paths['k1_k2_summary.csv'])
    values = np.sort(readout_table.alpha.unique())
    assert len(values) == ALPHA_BETA_GRID
    assert np.array_equal(values, np.sort(readout_table.beta.unique()))
    assert len(readout_table) == len(NORMS) * ALPHA_BETA_GRID**2
    assert len(depth_table) == len(NORMS) * (MAX_DEPTH + 1)**2
    assert not readout_table.duplicated(['norm','alpha','beta']).any()
    assert not depth_table.duplicated(['norm','k1','k2']).any()
    readout_planes, depth_planes, rate_roots = {}, {}, {}
    for norm in NORMS:
        r = readout_table.loc[readout_table.norm.eq(norm)]
        d = depth_table.loc[depth_table.norm.eq(norm)]
        readout_planes[norm] = {}
        depth_planes[norm] = {}
        for target, table, x, y, xvalues, yvalues in [
            (readout_planes[norm],r,'alpha','beta',values,values),
            (depth_planes[norm],d,'k1','k2',range(MAX_DEPTH+1),range(MAX_DEPTH+1))]:
            for field, column in [('high','high'),('count','n_stable'),('near','near_fold')]:
                target[field] = table.pivot(index=y,columns=x,values=column).reindex(index=yvalues,columns=xvalues).to_numpy()
            target['high'] = np.where(target['near'],np.nan,target['high'])
        rate_roots[norm] = [selected_equilibria.loc[selected_equilibria.norm.eq(norm) &
            selected_equilibria.condition.eq(kind)].sort_values('r').to_dict('records') for kind in KINDS]

assert selected_equilibria.residual.max() < 1e-10
for row in selected_equilibria.to_dict('records'):
    residual = abs(float(drift_and_derivative(row['r'],row['norm'],int(row['k1']),int(row['k2']),
        row['alpha'],row['beta'],row['e1'],row['e2'])[0]))
    assert residual < 1e-10
curve_records = []
for norm in NORMS:
    for kind,pair in zip(KINDS,condition_pairs(norm)):
        for r,drift in zip(np.linspace(0.,1.,1800),drift_and_derivative(np.linspace(0.,1.,1800),norm,*pair)[0]):
            curve_records.append(dict(norm=norm,condition=kind,k1=pair[0],k2=pair[1],r=r,drift=drift))
pd.DataFrame(curve_records).to_csv(DATA/'reputation_drift_curves.csv',index=False)
print('Loaded 12 curve conditions, four readout planes, and four depth planes.')


Loaded 12 curve conditions, four readout planes, and four depth planes.


## Drawing functions

Filled roots are stable in the scalar flow and open roots unstable; hatching denotes bistability. Colors distinguish no inertia, one-step inertia $(1,1)$ and the selected multi-step depths.

In [4]:
def format_axes(ax, labels, xlabel='', ylabel=''):
    if labels:
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
    else:
        ax.xaxis.set_major_formatter(NullFormatter())
        ax.yaxis.set_major_formatter(NullFormatter())
        ax.tick_params(labelbottom=False, labelleft=False)
        ax.xaxis.get_offset_text().set_visible(False)
        ax.yaxis.get_offset_text().set_visible(False)

def draw_phase_strip(ax, norm):
    ax.set(xlim=(-0.035, 1.035), ylim=(-0.48, 2.48))
    ax.axis('off')
    for y, kind, pair, rows in zip((2.0, 1.0, 0.0), KINDS, condition_pairs(norm), rate_roots[norm]):
        color = COLORS[kind]
        ax.plot([0, 1], [y, y], color=color, lw=0.85, alpha=0.38, zorder=1)
        roots = [row['r'] for row in rows]
        bounds = [0.0, *roots, 1.0]
        for left, right in zip(bounds[:-1], bounds[1:]):
            width = right - left
            if width < 0.045:
                continue
            center = 0.5 * (left + right)
            length = min(0.105, 0.45 * width)
            velocity = float(drift_and_derivative(center, norm, *pair)[0])
            if abs(velocity) < 1e-13:
                continue
            start, end = (center - length / 2, center + length / 2)
            if velocity < 0:
                start, end = (end, start)
            ax.annotate('', xy=(end, y), xytext=(start, y), arrowprops=dict(arrowstyle='-|>', color=color, lw=1.3, mutation_scale=9, shrinkA=0, shrinkB=0), zorder=3)
        for row in rows:
            ax.plot(row['r'], y, marker='D' if row['stability'] == 'near_fold' else 'o', ms=4.7, mfc=color if row['stability'] == 'stable' else 'white', mec=color, mew=1.1, linestyle='None', zorder=5)

def draw_rates(fig, cell, norm, labels=True, letter=None):
    inner = cell.subgridspec(2, 1, height_ratios=[0.25, 1.0], hspace=0.04)
    phase = fig.add_subplot(inner[0, 0])
    ax = fig.add_subplot(inner[1, 0])
    r = np.linspace(0.0, 1.0, 1800)
    series = []
    for kind, pair in zip(KINDS, condition_pairs(norm)):
        drift, _ = drift_and_derivative(r, norm, *pair)
        series.append(drift)
        ax.plot(r, drift, color=COLORS[kind], lw=1.8, zorder=3)
    ax.axhline(0.0, color='#303030', lw=0.8, zorder=2)
    low, high = (min(float(np.min(series)), 0.0), max(float(np.max(series)), 0.0))
    span = max(high - low, 0.04)
    ax.set(xlim=(-0.035, 1.035), ylim=(low - 0.08 * span, high + 0.08 * span), xticks=[0.0, 0.5, 1.0])
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.grid(color='#D9D9D9', lw=0.45, alpha=0.45, zorder=0)
    format_axes(ax, labels, 'Good reputation, $r_{\\rm DISC}$', 'Reputation drift, $\\dot r_{\\rm DISC}$')
    draw_phase_strip(phase, norm)
    add_panel_title(phase, norm, letter, labels)
    if labels:
        k1, k2 = MULTI_DEPTHS[norm]
        ax.text(0.04, 0.07, 'Multi-step: $(%d,%d)$' % (k1, k2), transform=ax.transAxes, color=COLORS['multi'], fontsize=8.5, bbox=dict(facecolor='white', edgecolor='none', alpha=0.88, pad=1.0))
    return ax

def apply_continuous_hatching(ax, mask, hatch='///', edge='#777777'):
    if np.any(mask):
        contour = ax.contourf(values, values, mask.astype(float), levels=[0.5, 1.5], colors='none', hatches=[hatch])
        for collection in getattr(contour, 'collections', [contour]):
            collection.set_edgecolor(edge)
            collection.set_linewidth(0.0)
        if np.any(~mask):
            ax.contour(values, values, mask.astype(float), levels=[0.5], colors=['#365F7D'], linewidths=0.9)

def draw_readout(ax, norm, labels=True, letter=None):
    plane = readout_planes[norm]
    ax.set_facecolor('#E0E0E0')
    image = ax.pcolormesh(values, values, plane['high'], vmin=0, vmax=1, cmap='Blues', shading='auto', rasterized=True)
    apply_continuous_hatching(ax, plane['count'] >= 2)
    apply_continuous_hatching(ax, plane['near'], hatch='...')
    ax.plot(0, 0, 's', ms=6, mfc='white', mec='#222222', mew=1.2, clip_on=False, zorder=5)
    ax.plot(1, 1, 'o', ms=6, mfc='none', mec=COLORS['one'], mew=1.7, clip_on=False, zorder=5)
    ax.set(xlim=(0, 1), ylim=(0, 1), xticks=[0, 0.5, 1], yticks=[0, 0.5, 1])
    format_axes(ax, labels, 'Tolerance, $\\alpha$', 'Strictness, $\\beta$')
    add_panel_title(ax, norm, letter, labels)
    return image

def draw_depth(ax, norm, labels=True, letter=None):
    plane = depth_planes[norm]
    ax.set_facecolor('#E0E0E0')
    image = ax.imshow(plane['high'], origin='lower', extent=(-0.5, MAX_DEPTH + 0.5, -0.5, MAX_DEPTH + 0.5), vmin=0, vmax=1, cmap='Blues', interpolation='nearest', aspect='auto')
    for j, i in zip(*np.where((plane['count'] >= 2) | plane['near'])):
        ax.add_patch(Rectangle((i - 0.5, j - 0.5), 1.0, 1.0, facecolor='none', edgecolor='#777777', lw=0, hatch='...' if plane['near'][j, i] else '///'))
    edges = np.arange(-0.5, MAX_DEPTH + 1.0, 1.0)
    ax.vlines(edges, -0.5, MAX_DEPTH + 0.5, color='white', lw=0.35, alpha=0.7)
    ax.hlines(edges, -0.5, MAX_DEPTH + 0.5, color='white', lw=0.35, alpha=0.7)
    for k1, k2, marker, color, size in [(0, 0, 's', '#222222', 6), (1, 1, 's', COLORS['one'], 7)]:
        ax.plot(k1, k2, marker=marker, ms=size, mfc='white', mec=color, mew=1.5, linestyle='None', zorder=6)
    ax.plot(k1, k2, marker=marker, ms=size, mfc='white', mec=color, mew=1.5, linestyle='None', zorder=6)
    ax.plot(*MULTI_DEPTHS[norm], marker='*', ms=12, mfc=COLORS['multi'], mec='white', mew=0.75, linestyle='None', zorder=7)
    ticks = np.arange(0, MAX_DEPTH + 1, 2)
    ax.set(xlim=(-0.5, MAX_DEPTH + 0.5), ylim=(-0.5, MAX_DEPTH + 0.5), xticks=ticks, yticks=ticks)
    format_axes(ax, labels, 'Tolerance depth, $k_1$', 'Strictness depth, $k_2$')
    add_panel_title(ax, norm, letter, labels)
    return image

def rate_legend():
    handles = [Line2D([0], [0], color=COLORS[kind], lw=1.8, label=label) for kind, label in zip(KINDS, ['Without inertia', 'One-step inertia (1,1)', 'Multi-step inertia'])]
    handles += [Line2D([0], [0], marker='o', ms=5, mfc=fill, mec='#333333', linestyle='None', label=label) for fill, label in [('#333333', 'Stable'), ('white', 'Unstable')]]
    return handles

def plane_legend(depth=False):
    handles = [Patch(facecolor='white', edgecolor='#777777', hatch='///', label='Bistable'), Patch(facecolor='white', edgecolor='#777777', label='Monostable')]
    handles += [Line2D([0], [0], marker='s' if depth else 'o', ms=6, mfc='white' if depth else 'none', mec=COLORS['one'], mew=1.5, linestyle='None', label='One-step inertia')]
    if depth:
        handles.append(Line2D([0], [0], marker='*', ms=10, mfc=COLORS['multi'], mec='white', linestyle='None', label='Multi-step inertia'))
    planes = depth_planes if depth else readout_planes
    if any((np.any(p['near']) for p in planes.values())):
        handles.append(Patch(facecolor='#E0E0E0', edgecolor='#777777', hatch='...', label='Near saddle-node'))
    return handles

def add_colorbar(fig, image, rectangle):
    bar = fig.colorbar(image, cax=fig.add_axes(rectangle), ticks=[0.0, 0.5, 1.0])
    bar.set_label('Equilibrium good reputation, $r^*$', labelpad=8)
    return bar

def make_full_figure():
    fig = plt.figure(figsize=(15.0, 10.8))
    grid = fig.add_gridspec(3, 4, left=0.065, right=0.93, bottom=0.12, top=0.94, wspace=0.38, hspace=0.78, height_ratios=[1.25, 1.0, 1.0])
    for i, norm in enumerate(NORMS):
        draw_rates(fig, grid[0, i], norm, letter='abcd'[i])
        ax = fig.add_subplot(grid[1, i])
        image = draw_readout(ax, norm, letter='efgh'[i])
        ax = fig.add_subplot(grid[2, i])
        draw_depth(ax, norm, letter='ijkl'[i])
    fig.legend(handles=rate_legend(), loc='center', bbox_to_anchor=(0.51, 0.667), ncol=5, frameon=False, fontsize=9, columnspacing=2.0)
    fig.legend(handles=plane_legend(depth=True), loc='lower center', bbox_to_anchor=(0.5, 0.022), ncol=5, frameon=False, fontsize=9, columnspacing=1.8)
    add_colorbar(fig, image, [0.95, 0.16, 0.012, 0.42])
    return fig

def add_panel_title(ax, norm, letter, labels):
    if labels:
        ax.set_title(NAMES[norm], fontsize=10, fontweight='bold', pad=7)
        if letter:
            ax.text(-.16,1.04,letter,transform=ax.transAxes,fontsize=12,fontweight='bold',va='bottom')


def save_figure(fig, stem):
    for extension in ('pdf','svg'):
        fig.savefig(FIGURES / f'{stem}.{extension}',bbox_inches='tight',pad_inches=.04,facecolor='white')


def make_single(row, norm):
    fig = plt.figure(figsize=(4.2,3.7))
    grid = fig.add_gridspec(1,1,left=.18,right=.85 if row else .97,bottom=.20,top=.90)
    if row == 0:
        draw_rates(fig,grid[0,0],norm)
    else:
        ax = fig.add_subplot(grid[0,0])
        image = (draw_readout if row == 1 else draw_depth)(ax,norm)
        fig.colorbar(image,cax=fig.add_axes([.89,.25,.035,.55]),ticks=[0,.5,1],label=r'$r^*$')
    return fig


## Population-level robustness to initial reputations (panel e)

Panel e compares mean long-run cooperation $\bar\rho$ over the 512 area-uniform initial compositions of Figure 4
when reputations are initialized once, at $t=0$, either at all-good ($r_I=1$) or at all-bad ($r_I=0$), and then
follow the strategy trajectory quasi-statically (warm-started relaxation of the auxiliary reputation flow at every
evaluation). This is a robustness protocol; the main model re-initializes reputations at all-good at every
composition. The data are produced by `../engine/initial_reputation.py` and copied to
`data/extended_data_figure2/population_initial_reputation.csv` (per-trajectory final states are stored next to the
engine results).

In [5]:
population = pd.read_csv(DATA / 'population_initial_reputation.csv')
assert len(population) == 24 and not population.duplicated(['norm', 'condition', 'initial_reputation']).any()
assert population.n_step_cap.eq(0).all() and population.max_final_strategy_residual.max() < 1e-4
POP = population.pivot_table(index=['norm', 'condition'], columns='initial_reputation', values='mean_cooperation')
POP['difference'] = POP['all_bad'] - POP['all_good']
display(POP.reindex([(n, c) for n in NORMS for c in ('none', 'one', 'multi')]).round(4))

initial_reputation  all_bad  all_good  difference
norm condition                                   
SC   none            0.6098    0.6098      0.0000
     one             0.5912    0.5912      0.0000
     multi           0.8651    0.8651      0.0000
SH   none            0.0000    0.0000      0.0000
     one             0.0000    0.0762     -0.0762
     multi           0.6910    0.6910      0.0000
SJ   none            0.0000    0.0000     -0.0000
     one             0.0000    0.2248     -0.2248
     multi           0.7063    0.6986      0.0077
SS   none            0.5434    0.5434      0.0000
     one             0.6960    0.6960     -0.0000
     multi           0.7410    0.7410      0.0000


## Figures

Extended Data Figure 2 (revised): panels a–d show the scalar reputation drift of homogeneous DISC populations and its
roots (filled: stable; open: unstable); panel e shows the population-level comparison. Extended Data Figure 9 contains
the former rows 2–3 of Extended Data Figure 2: the highest stable homogeneous-DISC reputation over the one-step
readout plane $(\alpha,\beta)$ and over the full-readout depth plane $(k_1,k_2)$; hatching marks bistability.
The combined figures follow the typography of the assembled manuscript figures (Helvetica, or TeX Gyre Heros/Arial
where Helvetica is unavailable; STIX mathematics). Single panels are also saved for assembly.

In [6]:
from matplotlib import font_manager
INSTALLED_FONTS = {f.name for f in font_manager.fontManager.ttflist}
SANS = next((n for n in ('Helvetica', 'TeX Gyre Heros', 'Arial') if n in INSTALLED_FONTS), 'DejaVu Sans')
PUB_RC = {'font.family': SANS, 'mathtext.fontset': 'stix',
          'font.size': 9, 'axes.labelsize': 9.5, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'axes.linewidth': 0.8,
          'axes.spines.top': False, 'axes.spines.right': False, 'pdf.fonttype': 42, 'svg.fonttype': 'none',
          'hatch.linewidth': 0.5}
NAVY = '#1F3864'
CONDITION_LABELS = {'without': 'Without inertia', 'one': 'One-step inertia', 'multi': 'Multi-step inertia'}

def pub_letter(ax, letter, x=-0.24, pad=4):
    """Panel letter on the baseline of the panel title (title pad in points)."""
    ax.annotate(letter, xy=(x, 1.0), xycoords='axes fraction', xytext=(0, pad), textcoords='offset points',
                fontsize=14, va='baseline', ha='left')

def boxed_legend(fig, handles, anchor, ncol, **kw):
    leg = fig.legend(handles=handles, loc='center', bbox_to_anchor=anchor, ncol=ncol, frameon=True, fancybox=False,
                     fontsize=8.5, borderpad=0.6, columnspacing=1.8, handlelength=2.0, **kw)
    leg.get_frame().set_edgecolor(NAVY); leg.get_frame().set_linewidth(0.9)
    return leg

def draw_rates_pub(fig, cell, norm, letter):
    inner = cell.subgridspec(2, 1, height_ratios=[0.27, 1.0], hspace=0.05)
    phase, ax = fig.add_subplot(inner[0, 0]), fig.add_subplot(inner[1, 0])
    r = np.linspace(0.0, 1.0, 1800); series = []
    for kind, pair in zip(KINDS, condition_pairs(norm)):
        drift, _ = drift_and_derivative(r, norm, *pair); series.append(drift)
        ax.plot(r, drift, color=COLORS[kind], lw=1.8, zorder=3)
    ax.axhline(0.0, color='#303030', lw=0.8, zorder=2)
    low, high = min(float(np.min(series)), 0.0), max(float(np.max(series)), 0.0); span = max(high - low, 0.04)
    ax.set(xlim=(-0.035, 1.035), ylim=(low - 0.08 * span, high + 0.14 * span), xticks=[0.0, 0.5, 1.0])
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.grid(color='#D9D9D9', lw=0.45, alpha=0.45, zorder=0)
    ax.set_xlabel(r'$r_{\rm DISC}$')
    if norm == 'SC': ax.set_ylabel(r'$\dot r_{\rm DISC}$')
    k1, k2 = MULTI_DEPTHS[norm]
    ax.text(1.0, 0.985, rf'$(k_1,k_2)=({k1},{k2})$', transform=ax.transAxes, ha='right', va='top', fontsize=8.5,
            color=COLORS['multi'])
    draw_phase_strip(phase, norm)
    phase.set_title(NAMES[norm], fontsize=10.5, pad=4)
    pub_letter(phase, letter, x=-0.24)
    return ax

def draw_population(ax):
    """Grouped bars: all-good (filled) and all-bad (hatched) initial reputations for each norm and condition."""
    width, gap = 0.36, 0.1
    centers = []
    for i, norm in enumerate(NORMS):
        for j, (kind, cond) in enumerate(zip(KINDS, ('none', 'one', 'multi'))):
            x0 = i * 3.6 + j * 1.0; centers.append((x0, f'{norm}\n{cond}'))
            good, bad = POP.loc[(norm, cond), 'all_good'], POP.loc[(norm, cond), 'all_bad']
            ax.bar(x0 - width / 2 - gap / 4, good, width, color=COLORS[kind], edgecolor=COLORS[kind], lw=0.8, zorder=3)
            ax.bar(x0 + width / 2 + gap / 4, bad, width, color='white', edgecolor=COLORS[kind], hatch='//////', lw=0.8, zorder=3)
            if max(good, bad) < 1e-3:
                ax.text(x0, 0.025, '0', ha='center', va='bottom', fontsize=6.6, color='#303030')
            elif abs(bad - good) > 1e-3:
                for x, v in ((x0 - width / 2 - gap / 4, good), (x0 + width / 2 + gap / 4, bad)):
                    ax.text(x, v + 0.025, f'{v:.3f}', ha='center', va='bottom', fontsize=6.6, rotation=90, color='#303030')
    ax.set_xticks([c[0] for c in centers])
    ax.set_xticklabels([('None' if 'none' in c[1] else 'One' if 'one' in c[1] else 'Multi') for c in centers], fontsize=8)
    for i, norm in enumerate(NORMS):
        ax.text(i * 3.6 + 1.0, -0.2, NAMES[norm], ha='center', va='top', fontsize=9.5, transform=ax.get_xaxis_transform())
        if i: ax.axvline(i * 3.6 - 0.8, color='#D0D0D0', lw=0.7, zorder=1)
    ax.set(xlim=(-0.7, 3 * 3.6 + 2.7), ylim=(0, 1.05), yticks=[0, 0.25, 0.5, 0.75, 1.0])
    ax.set_ylabel(r'Mean cooperation, $\bar\rho$')
    ax.grid(axis='y', color='#D9D9D9', lw=0.45, alpha=0.45, zorder=0)
    ax.tick_params(axis='x', length=0)

def make_ed2_figure():
    with plt.rc_context(PUB_RC):
        fig = plt.figure(figsize=(7.5, 6.9))
        top = fig.add_gridspec(1, 4, left=0.085, right=0.985, top=0.95, bottom=0.565, wspace=0.42)
        for i, norm in enumerate(NORMS): draw_rates_pub(fig, top[0, i], norm, 'abcd'[i])
        handles = [Line2D([0], [0], color=COLORS[k], lw=1.8, label=CONDITION_LABELS[k]) for k in KINDS]
        handles += [Line2D([0], [0], marker='o', ms=5, mfc='#222222', mec='#222222', ls='None', label='Stable'),
                    Line2D([0], [0], marker='o', ms=5, mfc='white', mec='#222222', ls='None', label='Unstable')]
        boxed_legend(fig, handles, (0.535, 0.475), 5)
        ax = fig.add_axes([0.085, 0.13, 0.9, 0.27]); draw_population(ax); pub_letter(ax, 'e', x=-0.075, pad=6)
        handles = [Patch(facecolor='#777777', edgecolor='#777777', label='All-good initial reputations'),
                   Patch(facecolor='white', edgecolor='#777777', hatch='//////', label='All-bad initial reputations')]
        boxed_legend(fig, handles, (0.535, 0.025), 2)
    return fig

def make_ed9_figure():
    with plt.rc_context(PUB_RC):
        fig = plt.figure(figsize=(7.5, 4.6))
        grid = fig.add_gridspec(2, 4, left=0.09, right=0.985, top=0.93, bottom=0.235, wspace=0.55, hspace=0.45)
        for i, norm in enumerate(NORMS):
            for row, (draw, letter) in enumerate(((draw_readout, 'abcd'[i]), (draw_depth, 'efgh'[i]))):
                ax = fig.add_subplot(grid[row, i]); image = draw(ax, norm, labels=True, letter=None)
                ax.set_title(NAMES[norm] if row == 0 else '', fontsize=10.5, fontweight='normal', pad=5)
                if i:
                    ax.set_ylabel('')
                pub_letter(ax, letter, x=-0.34, pad=5)
                ax.set_xlabel(r'Tolerance strength, $\alpha$' if row == 0 else r'Tolerance depth, $k_1$')
                if i == 0: ax.set_ylabel(r'Strictness strength, $\beta$' if row == 0 else r'Strictness depth, $k_2$')
                for line in ax.lines: line.set_markersize(line.get_markersize() * 0.8)
        cax = fig.add_axes([0.62, 0.065, 0.3, 0.022])
        bar = fig.colorbar(image, cax=cax, orientation='horizontal', ticks=[0, 0.25, 0.5, 0.75, 1.0])
        bar.set_label(r'Equilibrium good reputation, $r^*$', fontsize=8.5, labelpad=2); bar.ax.tick_params(labelsize=7.5)
        handles = [Line2D([0], [0], marker='s', ms=6, mfc='white', mec='#222222', mew=1.2, ls='None', label='Without inertia'),
                   Line2D([0], [0], marker='s', ms=6, mfc='white', mec=COLORS['one'], mew=1.4, ls='None', label='One-step inertia'),
                   Line2D([0], [0], marker='*', ms=9, mfc=COLORS['multi'], mec='white', ls='None', label='Multi-step inertia'),
                   Patch(facecolor='white', edgecolor='#777777', hatch='///', label='Bistable'),
                   Patch(facecolor='white', edgecolor='#777777', label='Monostable')]
        boxed_legend(fig, handles, (0.31, 0.075), 3, handletextpad=0.5)
    return fig

ED9_FIGURES = FIGURES.parent / 'extended_data_figure9'
ED9_FIGURES.mkdir(parents=True, exist_ok=True)

def save_to(fig, directory, stem, png=False):
    for extension in ('pdf', 'svg'):
        fig.savefig(directory / f'{stem}.{extension}', bbox_inches='tight', pad_inches=.04, facecolor='white')
    if png:
        fig.savefig(directory / f'{stem}.png', dpi=300, facecolor='white')

for row, group in enumerate(('drift', 'readout', 'depth')):
    for column, norm in enumerate(NORMS):
        fig = make_single(row, norm)
        save_to(fig, FIGURES if group == 'drift' else ED9_FIGURES, f'panel_{group}_{norm}')
        plt.close(fig)
with plt.rc_context(PUB_RC):
    fig, ax = plt.subplots(figsize=(7.2, 2.4)); draw_population(ax); fig.subplots_adjust(bottom=0.25)
    save_to(fig, FIGURES, 'panel_population_initial_reputation'); plt.close(fig)
fig = make_ed2_figure(); save_to(fig, FIGURES, 'Extended_Data_Figure2', png=True); plt.show(); plt.close(fig)
fig = make_ed9_figure(); save_to(fig, ED9_FIGURES, 'Extended_Data_Figure9', png=True); plt.show(); plt.close(fig)
print('Saved Extended Data Figures 2 and 9 (PDF, SVG, PNG) and 17 single panels.')

Saved Extended Data Figures 2 and 9 (PDF, SVG, PNG) and 17 single panels.
